# Qwen-Image 2.1 · A100 80GB

选择 A100 80GB，依次运行：**环境 → 模型与工作流 → 打开 ComfyUI → 下载结果**。

01 文生图；02 参考图；03/04 使用 LoRA。参考版先上传图片。默认 2K，下拉可选 1K；参考版还可选原图。

第 2 格下载工作流，同名直接覆盖。第 3 格点击新窗口链接并保持 Colab 连接；生成后在第 4 格下载单张或全部图片。使用 03/04 前先下载 LoRA。


In [ ]:
# @title 1. GPU 与环境
import os, re, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("请在 Colab 选择 A100 80GB GPU。")
gpu = torch.cuda.get_device_name(0)
memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu, f"{memory:.1f} GiB")
if "A100" not in gpu.upper() or memory < 70:
    raise RuntimeError("当前 GPU 不是 A100 80GB，请切换 runtime。")

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REVISION = 'c9d8a6e69c4b5ab7fa0f789e7b988c172cd31fc9'
def run(args):
    subprocess.run(args, cwd=COMFY_DIR if COMFY_DIR.exists() else None, check=True)
if COMFY_DIR.is_symlink():
    raise RuntimeError("ComfyUI 目录是软链接，请使用新的 runtime。")
fresh = not COMFY_DIR.exists()
if fresh:
    subprocess.run(["git", "clone", "--depth", "1", "--no-checkout", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)], check=True)
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI 目录不是 Git 仓库，请使用新的 runtime。")
origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=COMFY_DIR, text=True).strip()
if origin.rstrip("/").removesuffix(".git") != "https://github.com/Comfy-Org/ComfyUI":
    raise RuntimeError("现有 ComfyUI 不是官方仓库，请使用新的 runtime。")
current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=COMFY_DIR, text=True).strip()
if not fresh:
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=COMFY_DIR, text=True).strip()
    if changes:
        raise RuntimeError("现有 ComfyUI 有未提交修改，请使用新的 runtime。")
if fresh or current != COMFY_REVISION:
    run(["git", "fetch", "--depth", "1", "origin", COMFY_REVISION])
    run(["git", "checkout", "--detach", COMFY_REVISION])
skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for line in (COMFY_DIR / "requirements.txt").read_text().splitlines():
    name = re.split(r"[<>=!~\[]", line.strip(), maxsplit=1)[0].strip().lower()
    if name not in skip:
        lines.append(line)
requirements = Path("/content/qwen_comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "huggingface_hub", "hf_xet", "ipywidgets"])
print("ComfyUI:", COMFY_REVISION[:12])


In [ ]:
# @title 2. 模型与 GitHub 工作流
import shutil
from huggingface_hub import hf_hub_download

MODEL_REPO = 'Comfy-Org/Qwen-Image-2.1'
MODEL_REVISION = 'cb504a4090723e43f17ad01cec0359490e2de613'
MODEL_FILES = [{'file': 'diffusion_models/qwen_image_2.1_bf16.safetensors', 'size': 14230280616}, {'file': 'text_encoders/qwen3vl_8b_bf16.safetensors', 'size': 17534334616}, {'file': 'vae/qwen_image_2.1_vae_bf16.safetensors', 'size': 675509688}]
MODEL_DIR = COMFY_DIR / "models"

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

for path in [MODEL_DIR, COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)

def download_models(manifest, directory):
    missing = sum(item["size"] for item in manifest if not (directory / item["file"]).is_file() or (directory / item["file"]).stat().st_size != item["size"])
    if missing and shutil.disk_usage(directory).free < missing + 2 * 1024**3:
        raise RuntimeError("本地磁盘空间不足；Qwen 2.1 模型合计约 32.4 GB。")
    for item in manifest:
        target = directory / item["file"]
        local_directory(target.parent)
        if target.is_symlink():
            raise RuntimeError(f"模型文件是软链接：{target}")
        force = target.is_file() and target.stat().st_size != item["size"]
        result = Path(hf_hub_download(repo_id=MODEL_REPO, filename=item["file"], revision=MODEL_REVISION, local_dir=str(directory), force_download=force))
        if result.resolve() != target.resolve() or not target.is_file() or target.stat().st_size != item["size"]:
            raise RuntimeError(f"模型下载不完整或路径不匹配：{target}")
        print(target.name)

download_models(MODEL_FILES, MODEL_DIR)

# These controls belong to the Qwen notebook model/workflow cell.
WORKFLOW_REF = "main" # @param {type:"string"}
下载默认动漫一致性_LoRA = False # @param {type:"boolean"}

import hashlib, json, os, shutil, tempfile, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import quote, urlencode

WORKFLOW_REPO = "boltguo/colab"
WORKFLOW_FOLDER = "workflows/qwen-image-2.1"
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows"
LORA_DIR = MODEL_DIR / "loras"
LORA_REPO = "WarmBloodAban/Qwen-Image-2.1-LoRAs"
LORA_REVISION = "c4ab5473bfdf585fc19cfd1e280f79d2b0c79947"
LORA_FILE = "Qwen2.1_Anime_consistency.safetensors"
LORA_SIZE = 167830408
LORA_SHA256 = "0c171eb802ea8051b511f2d93c1743eeadd030316255aa98fa65d40809366752"
EXPECTED_LOADERS = {"UNETLoader": "qwen_image_2.1_bf16.safetensors",
                    "CLIPLoader": "qwen3vl_8b_bf16.safetensors",
                    "VAELoader": "qwen_image_2.1_vae_bf16.safetensors"}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def download_default_lora():
    local_directory(LORA_DIR)
    target = LORA_DIR / LORA_FILE
    if target.is_symlink():
        raise ValueError("LoRA 文件是软链接，未写入。")
    if target.exists():
        if not target.is_file() or target.stat().st_size != LORA_SIZE or sha256_file(target) != LORA_SHA256:
            raise ValueError("同名 LoRA 与默认版本不同；保留原文件，请先检查或另存。")
        print("默认 LoRA 已存在：", target)
        return
    if shutil.disk_usage(LORA_DIR).free < LORA_SIZE + 1024**3:
        raise RuntimeError("磁盘空间不足，未下载 LoRA。")
    result = Path(hf_hub_download(repo_id=LORA_REPO, filename=LORA_FILE,
                                 revision=LORA_REVISION, local_dir=str(LORA_DIR)))
    if result.resolve() != target.resolve() or target.is_symlink() or target.stat().st_size != LORA_SIZE or sha256_file(target) != LORA_SHA256:
        raise RuntimeError("LoRA 下载路径、大小或 SHA256 校验失败。")
    print("默认 LoRA 已下载：", target)

def validate_workflow_models(data):
    if not isinstance(data, dict) or not isinstance(data.get("nodes"), list) or not data["nodes"] or not isinstance(data.get("links"), list):
        raise ValueError("工作流 JSON 缺少 nodes 或 links。")
    for node in data["nodes"]:
        if not isinstance(node, dict) or not isinstance(node.get("type"), str):
            raise ValueError("工作流节点结构无效。")
        expected = EXPECTED_LOADERS.get(node["type"])
        values = node.get("widgets_values") or []
        if expected and (not isinstance(values, list) or not values or values[0] != expected):
            raise ValueError(f"{node['type']} 的模型文件名与本 notebook 下载映射不一致。")
    return data

def report_lora_requirements(name, data):
    for node in data["nodes"]:
        if node.get("type") != "LoraLoaderModelOnly" or node.get("mode", 0) == 4:
            continue
        values = node.get("widgets_values") or []
        filename = values[0] if isinstance(values, list) and values else None
        safe_name = isinstance(filename, str) and bool(filename) and Path(filename).name == filename and "\\" not in filename
        target = LORA_DIR / filename if safe_name else None
        if target is None or not target.is_file() or target.is_symlink():
            print(f"前置要求：{name} 的 LoRA 已启用；请先将 {filename!r} 放入 {LORA_DIR}，刷新模型列表并选择。不会自动旁路。")

def github_json(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Qwen-Colab-Workflow-Downloader", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read(2_000_001)
        if len(body) > 2_000_000 or "text/html" in response.headers.get("Content-Type", "").lower():
            raise ValueError("GitHub 返回网页或文件超过 2 MB，未保存。")
    try:
        return json.loads(body)
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("GitHub 返回内容不是 JSON；未保存。") from error

def install_workflow(name, data):
    if not isinstance(name, str) or not name.lower().endswith(".json") or Path(name).name != name or "\\" in name:
        raise ValueError("工作流文件名无效。")
    validate_workflow_models(data)
    local_directory(WORKFLOW_DIR)
    target = WORKFLOW_DIR / name
    if target.is_symlink() or (target.exists() and not target.is_file()):
        raise ValueError("目标工作流是软链接或目录，未写入。")
    fd, temporary = tempfile.mkstemp(prefix=".qwen-workflow-", dir=WORKFLOW_DIR)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
            file.write("\n")
        os.replace(temporary, target)
        print("工作流已下载：", name)
        report_lora_requirements(name, data)
        return True
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def report_github_failure(error, name=None):
    label = f"工作流 {name}" if name else "GitHub 工作流目录"
    if isinstance(error, urllib.error.HTTPError) and error.code == 404:
        print(f"{label} 返回 404：可能尚未推送到 GitHub，或 WORKFLOW_REF={WORKFLOW_REF!r} 不存在。推送后重跑第 2 格；保留当前工作流。")
    else:
        print(f"{label} 下载失败：{error}。未保存错误页面，保留当前工作流。")

def download_github_workflows():
    ref = WORKFLOW_REF.strip()
    if not ref:
        raise ValueError("WORKFLOW_REF 不能为空。")
    api = f"https://api.github.com/repos/{WORKFLOW_REPO}/contents/{WORKFLOW_FOLDER}?" + urlencode({"ref": ref})
    try:
        entries = github_json(api)
        if not isinstance(entries, list):
            raise ValueError("GitHub 目录响应不是文件列表。")
    except (urllib.error.URLError, OSError, ValueError) as error:
        report_github_failure(error)
        return
    entries = [entry for entry in entries if isinstance(entry, dict) and entry.get("type") == "file" and str(entry.get("name", "")).lower().endswith(".json")]
    if not entries:
        print("该 GitHub 目录目前没有 JSON 工作流；推送后重跑第 2 格。")
    for entry in sorted(entries, key=lambda entry: entry["name"]):
        name = entry["name"]
        try:
            if not isinstance(name, str) or Path(name).name != name or "\\" in name:
                raise ValueError("GitHub 文件名无效。")
            url = f"https://raw.githubusercontent.com/{WORKFLOW_REPO}/{quote(ref, safe='')}/{WORKFLOW_FOLDER}/{quote(name, safe='')}"
            install_workflow(name, github_json(url))
        except (urllib.error.URLError, OSError, ValueError) as error:
            report_github_failure(error, name)

local_directory(WORKFLOW_DIR)
if 下载默认动漫一致性_LoRA:
    download_default_lora()
else:
    print("未选择下载 LoRA；03/04 已启用 LoRA，使用前需准备 models/loras/" + LORA_FILE + "，不会自动回退。")
download_github_workflows()


In [ ]:
# @title 3. 启动并打开 ComfyUI
import json, subprocess, sys, time, urllib.request
from pathlib import Path
from google.colab import output
from html import escape
from urllib.parse import urlsplit
from IPython.display import HTML, display

COMFY_DIR = Path("/content/ComfyUI")
BASE = "http://127.0.0.1:8188"
LOG = Path("/content/qwen_comfyui.log")
def get_json(route):
    with urllib.request.urlopen(BASE + route, timeout=3) as response:
        return json.load(response)
def ready():
    try:
        get_json("/system_stats")
        return True
    except Exception:
        return False

LAUNCHER = Path("/content/qwen_comfy_userdata_compat.py")
LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases for decoded proxy paths."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; compatibility aliases were not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            # Preserve the original user checks, safe response headers,
            # atomic writes, overwrite flags, and deletion semantics.
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI source directory is missing or invalid.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''
LAUNCHER.write_text(LAUNCHER_SOURCE)
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8188)")
if not isinstance(proxy_url, str) or not proxy_url.startswith("https://"):
    raise ValueError("Colab 未返回 HTTPS 代理链接。")
proxy_parts = urlsplit(proxy_url)
proxy_origin = f"{proxy_parts.scheme}://{proxy_parts.netloc}"

if not ready():
    with LOG.open("w") as log:
        qwen_process = subprocess.Popen([sys.executable, str(LAUNCHER), "--listen", "127.0.0.1", "--port", "8188", "--output-directory", str(COMFY_DIR / "output"), "--enable-cors-header", proxy_origin, "--disable-all-custom-nodes", "--disable-partner-nodes"], cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.time() + 600
    while time.time() < deadline:
        if qwen_process.poll() is not None:
            print(LOG.read_text(errors="replace")[-8000:])
            raise RuntimeError("ComfyUI 启动失败，请查看上方日志。")
        if ready():
            break
        time.sleep(2)
    else:
        raise TimeoutError("ComfyUI 10 分钟内未就绪。")
registered = get_json("/object_info")
if not {"TextEncodeQwenImage21", "QwenImage21Cache", "CustomCombo", "ComfyMathExpression", "ResolutionSelector"}.issubset(registered):
    raise RuntimeError("缺少 Qwen 或内置尺寸节点，请重跑第 1 格安装固定版本，再运行第 3 格。日志：/content/qwen_comfyui.log。")
display(HTML(f'<a href="{escape(proxy_url, quote=True)}" target="_blank" rel="noopener">新窗口打开 ComfyUI</a>'))
print("打开工作流 01–04，或拖入 JSON；参考图片在图片节点上传。")
print("默认 2K，可下拉切换。03/04 需先下载 LoRA。")
print("生成后运行第 4 格下载；重跑该格刷新列表。")


In [ ]:
# @title 4. 下载图片与工作流
import json, os, tempfile, zipfile
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from google.colab import files

OUTPUT_DIR = Path("/content/ComfyUI/output")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
def result_files(directory):
    directory = Path(directory)
    if directory.is_symlink():
        raise ValueError("输出目录是软链接，请核对保存目录。")
    root = directory.resolve()
    results = []
    for base, folders, names in os.walk(directory, followlinks=False):
        folders[:] = [name for name in folders if not (Path(base) / name).is_symlink()]
        for name in names:
            path = Path(base) / name
            if path.suffix.lower() in IMAGE_EXTENSIONS and not path.is_symlink() and path.is_file() and path.resolve().is_relative_to(root):
                results.append(path)
    return sorted(results, key=lambda path: path.stat().st_mtime, reverse=True)

def make_result_zip(selected, directory, archive):
    selected, directory, archive = Path(selected), Path(directory), Path(archive)
    if selected not in result_files(directory) or not selected.resolve().is_relative_to(directory.resolve()):
        raise ValueError("请选择列表中的图片。")
    if archive.is_symlink() or archive.resolve().is_relative_to(directory.resolve()):
        raise ValueError("ZIP 须保存在 output 外，且不能是软链接。")
    workflow = None
    try:
        with Image.open(selected) as image:
            embedded = image.info.get("workflow")
        if embedded:
            data = json.loads(embedded) if isinstance(embedded, str) else embedded
            if isinstance(data, dict) and isinstance(data.get("nodes"), list) and isinstance(data.get("links"), list):
                workflow = data
    except Exception as error:
        print("无法读取内嵌工作流，仍下载图片：", error)
    fd, temporary = tempfile.mkstemp(prefix=".qwen-result-", suffix=".zip", dir=archive.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as bundle:
            if selected.is_symlink() or not selected.is_file() or not selected.resolve().is_relative_to(directory.resolve()):
                raise ValueError("图片路径已变化，未完成打包。")
            bundle.write(selected, arcname=selected.name)
            if workflow is not None:
                bundle.writestr("workflow.json", json.dumps(workflow, ensure_ascii=False, indent=2))
        os.replace(temporary, archive)
        return archive
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def make_all_images_zip(directory, archive):
    directory, archive = Path(directory), Path(archive)
    results = result_files(directory)
    if not results:
        raise ValueError("输出目录及子目录内暂无 PNG、JPG、JPEG 或 WEBP 图片。")
    if archive.is_symlink() or archive.resolve().is_relative_to(directory.resolve()):
        raise ValueError("ZIP 须保存在 output 外，且不能是软链接。")
    fd, temporary = tempfile.mkstemp(prefix=".qwen-output-", suffix=".zip", dir=archive.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as bundle:
            for path in results:
                if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(directory.resolve()):
                    raise ValueError("图片路径已变化，未完成打包。")
                bundle.write(path, arcname=path.relative_to(directory).as_posix())
        os.replace(temporary, archive)
        return archive
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

results = result_files(OUTPUT_DIR)
print("输出目录：", OUTPUT_DIR, "；找到图片：", len(results))
if not results:
    print("暂无图片。生成后重跑本格；已有结果请核对输出目录。")
else:
    selector = widgets.Dropdown(options=[(str(path.relative_to(OUTPUT_DIR)), str(path)) for path in results], description="图片", layout=widgets.Layout(width="95%"))
    button = widgets.Button(description="下载所选图片＋工作流")
    all_button = widgets.Button(description="下载全部图片")
    status = widgets.Output()
    def download(_):
        with status:
            status.clear_output()
            try:
                archive = make_result_zip(selector.value, OUTPUT_DIR, Path("/content/qwen_result.zip"))
                files.download(str(archive))
            except Exception as error:
                print("下载失败：", error)
    def download_all(_):
        with status:
            status.clear_output()
            try:
                archive = make_all_images_zip(OUTPUT_DIR, Path("/content/qwen_all_output_images.zip"))
                print("全部图片已打包。")
                files.download(str(archive))
            except Exception as error:
                print("下载全部图片失败：", error)
    button.on_click(download)
    all_button.on_click(download_all)
    display(selector, widgets.HBox([button, all_button]), status)


Built with Qwen. [模型许可](https://huggingface.co/Qwen/Qwen-Image-2.1/blob/main/LICENSE)。
